# Step 03 — is the study's own batch effect already removed?

**Data type: RNA_array** (GSE65391). **Reads:** `step02_expression.rds`. **Writes:** `step03_expression.rds`.

The arrays were run in two batches. The study design gives a direct test: **the same healthy blood
samples were run in both batches** (`set = Technical_Replicate`). Two arrays made from one blood sample
differ only by batch and by measurement noise. If a batch effect remained, the batch 2 copy of every
pair would sit higher, or lower, than its batch 1 copy in the same genes.

This comes before any site exists, and before we plant a site effect of our own.

In [1]:
source("../src/paths.R")
s2   <- readRDS(art("step02_expression.rds"))
E    <- s2$E[s2$expressed, ]
meta <- s2$meta
table(set = meta$set, batch = meta$batch)

                     batch
set                     1   2
  Technical_Replicate  16   8
  Test                268  55
  Training            554  95

## Pair the replicates

A pair is one healthy child at one visit, measured in both batches.

In [2]:
h     <- meta[meta$disease == "Healthy", ]
pairs <- split(rownames(h), paste(h$subject, h$visit))
pairs <- pairs[vapply(pairs, function(p) setequal(h[p, "batch"], c("1", "2")), logical(1))]
b1 <- vapply(pairs, function(p) p[h[p, "batch"] == "1"][1], "")
b2 <- vapply(pairs, function(p) p[h[p, "batch"] == "2"][1], "")
length(pairs)

[1] 23

## Test 1: does batch 2 move genes in a consistent direction?

For each gene: the batch 2 minus batch 1 difference in every pair, and a paired t statistic for the
mean difference. A residual batch effect would give thousands of genes with |t| above 5.

In [3]:
d  <- E[, b2] - E[, b1]
mu <- rowMeans(d)
tt <- mu / (apply(d, 1, sd) / sqrt(ncol(d)))
c(genes = nrow(d), median_abs_shift = round(median(abs(mu)), 3),
  largest_abs_shift = round(max(abs(mu)), 3), genes_abs_t_above_5 = sum(abs(tt) > 5))

genes    median_abs_shift   largest_abs_shift genes_abs_t_above_5 
           8825.000               0.048               0.496               0.000

## Test 2: are the two copies closer to each other than to other children?

In [4]:
cc <- cor(E[, b1])
c(replicate_pairs = round(mean(diag(cor(E[, b1], E[, b2]))), 3),
  different_children_same_batch = round(mean(cc[upper.tri(cc)]), 3))

replicate_pairs different_children_same_batch 
                        0.969                         0.928

## Test 3: how much of the main axes of variation is batch?

Principal component analysis (PCA) on all samples; for the first five components, the share of their
variance explained (R²) by batch and by disease.

In [5]:
pc <- prcomp(t(E - rowMeans(E)), rank. = 5)
r2 <- function(y, f) summary(lm(y ~ f))$r.squared
data.frame(component  = paste0("PC", 1:5),
           variance   = round((pc$sdev^2 / sum(pc$sdev^2))[1:5], 3),
           R2_batch   = round(apply(pc$x, 2, r2, f = meta$batch), 3),
           R2_disease = round(apply(pc$x, 2, r2, f = meta$disease), 3), row.names = NULL)

component,variance,R2_batch,R2_disease
<chr>,<dbl>,<dbl>,<dbl>
PC1,0.192,0.002,0.100
PC2,0.061,0.002,0.030
PC3,0.057,0.008,0.007
PC4,0.045,0.050,0.003
PC5,0.038,0.059,0.098


## Drop the technical replicates

They are second copies of samples already in the table. One array per blood sample remains.

In [6]:
keep <- meta$set != "Technical_Replicate"
table(disease = meta$disease[keep], batch = meta$batch[keep])
saveRDS(list(E = s2$E[, keep], expressed = s2$expressed, genes = s2$genes, meta = meta[keep, ]),
        art("step03_expression.rds"))
cat("wrote", art("step03_expression.rds"), "\n")

         batch
disease     1   2
  Healthy  16  32
  SLE     806 118

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step03_expression.rds 


## Findings

The study's batch effect has already been removed from the deposited values. Across 23 replicate
pairs, no gene moves consistently between batches. Replicates correlate with each other more than with
other children, and batch explains at most 6% of any of the first five components, less than
disease explains of the first. We do not correct for study batch. 972 samples remain.